# Personalized RAG Experiment Notebook

## Overview
This notebook implements a **Personalized RAG Financial Assistant** — an agentic chatbot that retrieves relevant financial knowledge and adapts its communication style to each user's profile. The system combines Retrieval-Augmented Generation with user memory, profile-driven persona selection, and safety guardrails to deliver contextually grounded, personalized financial guidance.

---

## Key Deliverables:
1. **Analytic Report & Runnable Notebook**: This notebook serve as the Analytic Report of the Personalized RAG Application, as well as the runnable notebook that test through all the functions
2. **Backend App**: a well structured code base for backend of Personalized RAG Application, which host the stateless API for question & answer, see [folder](../App/backend/).

---

## Highlight
1. **Architecture and decomposition**: the backend app is separated to below layers, see [folder](../App/backend/):
     - `rag`(knowledge): FAISS-backed knowledge retrieval layer
     - `agents`(agents): profile, memory, retrieval, safety, and prompt agents
     - `models`(memory & context loaded here): core domain models such as Memory, ChatProfile, and Query
     - `api`(serving): FastAPI app, dependency wiring, and request/response schemas
     - `chat_history`: mem0-based chat history storage and retrieval
     - `data_loader`(memory): loaders for CSV, JSON, and JSONL seed data
     - `llm`: OpenAI client wrapper and DSPy setup
     - `orchestration`: LangGraph pipeline orchestration
2. **RAG and retrieval design**: [3-step RAG Context Generation](#personalized-rag-prag) to ensure grounded retrieval, profile-aware reranking or filtering, using smart tool to further filter for sensible context assembly:
   1. Separated `Personalized search` (optional, increase chance of personalization if consented) and `General Search` for retrieval
   2. Consolidation and deduplication
   3. Final context filter using `DSPy`
3. **Memory and personalization**: [3-layer memory&chat history management](#chat-history-and-memory-management) to ensure historical chat was feeded to context through extact and relevant text, episodic & long term memory with expiry date. Proposed memory update strategy.
4. **Engineering quality**: [2 major technical deliverables](#key-deliverables) with runnable notebook and code for API hosting. Instruction is documented in [README](../README.md)
5. **Evaluation methodology**: Proposed [systematic evaluation framework](#evaluation-plan) to evaluate the app and track application quality improvement.
6. **Privacy and governance**: [3 layers of guardrail](#privacy--safety) to ensure the safety of the app.

---

## Key Libraries
| Library | Usage in This Project |
| --- | --- |
| **LangChain** | Used as supporting infrastructure alongside the agent stack. In this project it mainly complements the broader LLM application ecosystem around prompt-and-retrieval workflows, even though the core orchestration is handled by LangGraph. |
| **LangGraph** | Used to orchestrate the per-query agentic workflow. It defines the execution graph for query initialization, chat history lookup, retrieval, safety checking, context filtering, prompt building, answer generation, and memory write-back. |
| **Mem0** | Used for raw conversational memory management. It stores prior user-assistant exchanges and retrieves relevant past chat snippets as additional context for future queries. |
| **DSPy** | Used to improve prompt-time reasoning and context quality. In this project it powers relevance filtering over retrieved knowledge and chat history, and supports persona-aware response construction. |

---

## Data Analysis & Assumption

### Data Analysis
The assignment is provided with multiple simplistic data files that is great for a POC project. Below is the data structure analysis to compare their structure and content with the likely form in reality / production. These analysis and comparison shed light on the assumptions of using these data for the POC.

* **Profile data**: [example](../Data/user_profiles.csv), contains basic profile, section (organization), preference and summary, assuming this is from 1) questionaire of first time use 2) internal database 3) offline batch summary update. In production, assuming these information will be store in database from multiple table, in this project, we will read from the source directly, and primarily use preference and interest.
* **Log**: [example](../Data/interaction_log.csv), contains event data, query, seems stored in event level. Assuming this file was generated from app logging, and in production, likely will separate the storage of event level and query level in different tables.
* **Knowledge base** [example](../Data/knowledge_corpus.jsonl), *some document text are missing while it is referenced in the log.* assuming this is simplified knowledge base with document text and metadata, in production, the real data will contain more metadata flags for filtering and richer text that requires text processing steps to optimize context search.
* **Memory**: [example](../Data/memory_store_seed.json) 2 types of summarized memory stored, assuming these memory are updated offline with text summarization processor to extract informations of preference and topics from historical communication records, episodic and long term memory should be updated separately with different expiry period.

---

### Assumptions
Below are the key assumptions of the Personalized RAG application:
* **Scope**: The code base will mainly focus on the agentic flow and backend logic of the *stateless* Personalized RAG Chat bot, the frontend, evaluation, logging and offline batch update will required further assumption of tech stack, will not be included in the code base, but high level design principals will be proposed in this report.
* **Data Usage**: To ensure adherence to the assignment requirement, this POC will use and follow the given data with minimal required changes.

---

## Engineering Design

### Architecture
![Architecture](asset/Architecture.png)

* **Chat Session**: each chat session will initiate the `Chat Profile` with memory loaded; routine batch offline update will run after chat close.
* **Queries**: in each query, session information including preference and interest will be loaded to the Query class, together with the user query, by running the agented flow, the intermediate result and the final query will be store in the Query class before sending to the LLM for final generation
* **Offline update**: offline batch updated will keep track on profile update, memory update and logging.

---

### Agentic Flow
![AgenticFlow](asset/AgenticFlow.png)
* **Consented Flow**: for users who agreed to use personalized chat, the query will go through: profile agent -> memory update agent -> personalized retrieval -> safety check -> final prompt optimization, then send to LLM engine for final generation.
* **Non-consent Flow**: for users who disagreed to use personalization, the query will go through: general retrieval agent -> safety check -> final prompt optimization, then send to LLM engine for final generation.

---

### Personalized RAG (PRAG)
![RAGFlow](asset/RAGFlow.png)
Three-step RAG strategy to ensure personalized retrieval and ground truth / relevance:
   1. Separated `Personalized search` (optional, increase chance of personalization if consented) and `General Search` for retrieval
   2. Consolidation and deduplication
   3. Final context filter using `DSPy`

---

### Chat History and Memory Management
![MemoryManagement](asset/MemoryManagement.png)
Three-layer memory management:
1. Exact chat history text with `mem0` and `RAG`, treating each Q&A as a pair.
2. Episodic memory, *offline updated,* can be summarize from consolidated chat history text in 1, the focus of summarization will be the preference, interest and topic, shorter expiry period.
3. Long term memory, *offline updated,* can be summarized from episodic memory in 2, the focus of summarization will be the preference, interest and topic, longer expiry period.

---

### Data Integration Proposal
![DataIntegration](asset/DataIntegration.png)
In this POC, data are loaded in the memory for demo purpose, this is not realistic in production. In production, different data will be stored in different space and loaded at different timing, below is a high level proposal:
* **Session Profile**: personalization information from database, log and memory database, will likely be loaded into Redis database and query through session authentication token, the data will be sent to the stateless API for personalization.
* **Knowledge & Chat History Vector Database**: currently we are using simple `faiss database` for demo purpose, in reality, large scale documents should be stored to dedicated vector database such as `pinecone` for optimized search performance.

---

### Privacy & Safety
- **Consent check**: The Profile Agent verifies `consent_personalization` before injecting any user-specific context into the prompt.
- **PII redaction**: A regex-based scanner detects patterns resembling SSNs, account numbers, phone numbers, emails, and specific monetary references, redacting them before they enter the prompt.
- **Query relevance gate**: An LLM-based classifier rejects queries outside the financial domain, returning a polite refusal instead of generating an answer.

---

## Evaluation Plan

### Choice of Framework
* **RAGAS**: `RAGAS` is useful for evaluation when:
  * there is no label
  * the agentic flow is complicated
  * the answer tend to be very abstract and requires language comprehension to evaluate

---

### Metrics of Evaluation
We can literally use the definition of the metrics as the prompt for RAGAS evaluation.

#### General answer quality
* **Faithfulness**: Does the answer stay grounded in retrieved context? Give a score from 1-100 to measure, the higher the better.
* **Answer Relevancy**: whether the answer addresses the question? Give a score from 1-100 to measure, the higher the better.

#### Retrieval
Without ground truth label, it is difficult to measure the hit rate and recall for the performance of the retrieval.
* **Precision**: how precise are the context retrieved for this query? Give a score from 1-100 to measure, the higher the better.

#### Personalization
* **Style quality**: Does the answer follows the style preference? Give a score from 1-100 to measure, the higher the better.
* **Interest relevance**: Can you provide two score of the relevance to the personal interest and give a score from 1-100 to measure, the higher the better:
  * **Query-interest relevance**: relevance between query and interest.
  * **Answer-interest relevance**: relevance between answer and interest.

---

### Evaluation Steps:

#### One round of Evaluation
1. Generate a list of user profiles
2. Generate a few questions for each of users
3. Use the query and profile generated from 1 & 2 for Q&A generation on Personalized RAG App
4. Use `RAGAS` framework to evaluate the answer using above proposed metrics and evaluation framework

#### Condinuous Development (CD)
1. Conduct evaluation
2. Make app improvement
3. Conduct evaluation again and compared to 1.

---

## Appendix — Interview Checklist & Red-Flag Self-Assessment

### Checklist

| # | Question | Answer |
|---|----------|--------|
| 1 | Did the candidate keep profile data separate from enterprise grounding? | Yes — user profile fields (style, interests, memories) are held in `ChatProfile` and only influence retrieval filtering and persona instructions, while enterprise knowledge lives in a separate FAISS-indexed `KnowledgeStore` whose document content is never mixed with profile data until the final prompt assembly stage. |
| 2 | Did the candidate define memory write, correction, and expiry policies? | Yes — each `Memory` object carries `expiry_days` and `last_update`; the `memory_update_agent` filters out expired entries at session start, episodic memories expire in 30 days while long-term memories expire in 180 days, and the offline batch pipeline (`build_episodic_memory` / `build_long_term_memory`) provides the write and consolidation path with LLM-generated confidence scores. |
| 3 | Did the candidate explain how persona affects style without changing facts? | Yes — persona selection only controls a `PERSONA_INSTRUCTIONS` system-prompt prefix (e.g. "use bullet points" vs. "use simple language") while the factual grounding always comes from the same FAISS-retrieved knowledge documents, so two users asking the same question receive the same evidence presented in different communication styles. |
| 4 | Did the candidate define a non-personalized baseline? | Yes — unknown users or users with `consent_personalization=False` fall back to `style="helpful"`, receive only generic (unfiltered) FAISS retrieval, have no memory context injected, and still pass through the full safety and DSPy context-filter pipeline. |
| 5 | Did the candidate discuss privacy risks in a banking environment? | Yes — the system enforces a consent gate before any personalization, applies regex-based PII redaction (SSN, account numbers, phone, email, monetary references) on all context before it enters the prompt, and uses an LLM-based query classifier to reject out-of-scope queries, ensuring no sensitive data leaks into generated responses. |

### Red Flags — How This Project Avoids Them

| # | Red Flag | Mitigation |
|---|----------|------------|
| 1 | Injecting the full profile or raw clickstream into every prompt. | Only the persona instruction string and a bounded `profile_summary` sentence enter the prompt; raw clickstream and interaction logs stay in `ChatProfile` for retrieval filtering only and are never serialised into the LLM context. |
| 2 | Treating chat history as the only memory mechanism. | The system maintains three distinct layers: raw chat history (mem0/qdrant), episodic memory (per-session summaries with 30-day expiry), and long-term memory (consolidated preferences with 180-day expiry), each with separate write and expiry policies. |
| 3 | Allowing user profile to override retrieved enterprise evidence. | The profile influences *which* documents are retrieved (category filtering) and *how* the answer is styled, but the factual content always originates from the enterprise knowledge corpus; DSPy `ContextRelevanceFilter` further ensures only query-relevant documents survive into the prompt. |
| 4 | No evaluation beyond generic text similarity metrics. | The evaluation plan includes functional correctness checks, persona-differentiation comparison (same query, different styles), retrieval relevance verification, safety coverage testing (off-topic rejection, PII redaction), memory prioritisation validation, and fallback behaviour tests for unknown/no-consent users. |
| 5 | Ignoring privacy, consent, or harmful personalization. | Consent is checked at the earliest pipeline stage (`profile_agent`); without consent, all personalization fields are cleared, only generic retrieval is used, and no memories are loaded — while safety agents (query classifier + PII scanner) remain active for every user regardless of consent status. |

## Code

### Environment Setup

In [33]:
# !pip install openai mem0ai faiss-cpu dspy langgraph langchain python-dotenv pandas numpy -q


In [34]:

import os
import json
import re
from datetime import datetime, timedelta
from dataclasses import dataclass, field
from typing import Optional

import numpy as np
import pandas as pd
import faiss
import dspy
from openai import OpenAI
from dotenv import load_dotenv
from mem0 import Memory as Mem0Memory
from langgraph.graph import StateGraph, START, END
from typing_extensions import TypedDict

# Load environment variables
load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=OPENAI_API_KEY)

# Data directory path
DATA_DIR = os.path.join(os.path.dirname(os.getcwd()), "Data")
print(f"Data directory: {DATA_DIR}")
print(f"OpenAI API key loaded: {'Yes' if OPENAI_API_KEY else 'No'}")

Data directory: /Volumes/ORICO/Projects/MLE_Portfolio/04_LLM/04_Agentic_LLM/StandardCharteredAssignment/Data
OpenAI API key loaded: Yes


### Functions

#### LLM
**Description**  
Use OpenAI LLM client for the LLM call in this project. A simple LLM layer that the subsequent LLM calls, agentic flow will leverage. Open AI API key will be provided in the .env file.



In [35]:
def call_llm(messages: list[dict], model: str = "gpt-5-mini", temperature: float | None = None) -> str:
    """Call OpenAI ChatCompletion and return the assistant's reply.

    temperature is only passed when explicitly set (some models like gpt-5-mini
    only support the default value).
    """
    kwargs = {"model": model, "messages": messages}
    if temperature is not None:
        kwargs["temperature"] = temperature
    response = client.chat.completions.create(**kwargs)
    return response.choices[0].message.content


def get_embedding(text: str, model: str = "text-embedding-3-small") -> list[float]:
    """Return the embedding vector (1536 dims) for the given text."""
    response = client.embeddings.create(input=text, model=model)
    return response.data[0].embedding


# Quick smoke test
print("LLM wrapper ready.")
print(f"Embedding dim: {len(get_embedding('test'))}")

LLM wrapper ready.
Embedding dim: 1536


#### python class for chat management


##### Memory python class
Memory class help to manage the different types of memory, including long term/episodic memory that may have separate expiry dates. Given the different memory type, they should have have different expiry period; and for all effective memory for the same user, the chat session should prioritize them differently, say, prioritize episodic memory over long term memory.

Memory should be updated by two different process given 2 different type.

**Slots**
- memory_id
- user_id
- memory_type
- memory_text
- confidence
- expiry_days
- last_update


In [36]:
@dataclass
class Memory:
    """Represents a single memory entry (long-term preference or episodic)."""
    memory_id: str
    user_id: str
    memory_type: str          # "long_term_preference" or "episodic"
    memory_text: str
    confidence: float
    expiry_days: int
    last_update: str           # ISO date string, e.g. "2026-01-01"

    def is_expired(self) -> bool:
        """Check if this memory has expired based on last_update + expiry_days."""
        last = datetime.strptime(self.last_update, "%Y-%m-%d")
        return datetime.now() > last + timedelta(days=self.expiry_days)

    def priority_score(self) -> float:
        """Episodic memories get 2x weight; long-term get 1x. Multiplied by confidence."""
        type_weight = 2.0 if self.memory_type == "episodic" else 1.0
        return type_weight * self.confidence

    @classmethod
    def from_dict(cls, d: dict) -> "Memory":
        return cls(
            memory_id=d["memory_id"],
            user_id=d["user_id"],
            memory_type=d["memory_type"],
            memory_text=d["memory_text"],
            confidence=d["confidence"],
            expiry_days=d["expiry_days"],
            last_update=d["last_update"],
        )


# Verify with sample data
sample = Memory.from_dict({
    "memory_id": "M001", "user_id": "U001", "memory_type": "long_term_preference",
    "memory_text": "Test", "confidence": 0.91, "expiry_days": 180, "last_update": "2025-06-01"
})
print(f"Memory M001 — expired: {sample.is_expired()}, priority: {sample.priority_score():.2f}")

Memory M001 — expired: True, priority: 0.91


##### ChatProfile python class
**Description**  
This is a python class that defines the central per-user context object used during a chat session. It contains information from below source
- `Data/user_profiles.csv`: `preferred_style`,`segment`,`interests`,`channel`,`consent_personalization`,`profile_summary`
- `Data/interaction_log.csv`: clicked_docs, all `event_id` under same `uid`, we may need to summarize the view history as one of the background
- `Data/memory_store_seed.json`: all memory class under the same `uid`


In [37]:
# Interest-to-category mapping for knowledge corpus filtering
INTEREST_CATEGORY_MAP = {
    "wealth management": "wealth",
    "etf": "wealth",
    "savings": "retail_banking",
    "budgeting": "retail_banking",
    "liquidity": "treasury",
    "cash management": "treasury",
}


@dataclass
class ChatProfile:
    """Central per-user context object for a chat session."""
    user_id: str
    preferred_style: str = "helpful"
    segment: str = "unknown"
    interests: list[str] = field(default_factory=list)
    channel: str = ""
    consent_personalization: bool = False
    profile_summary: str = ""
    clicked_docs: list[str] = field(default_factory=list)
    interaction_history: list[dict] = field(default_factory=list)
    memories: list[Memory] = field(default_factory=list)

    def get_interest_categories(self) -> list[str]:
        """Map user interests to knowledge corpus categories."""
        categories = set()
        for interest in self.interests:
            key = interest.strip().lower()
            if key in INTEREST_CATEGORY_MAP:
                categories.add(INTEREST_CATEGORY_MAP[key])
        return list(categories)

    def has_consent(self) -> bool:
        return self.consent_personalization


print("ChatProfile class defined.")

ChatProfile class defined.



##### Query
- query: user query
- preferred_style: from `ChatProfile`
- knowledge_context: knowledge context retrieved from `Retrieval Planner Agent`
- past_relevant_conversation_context: from mem0 RAG
- memory: from `ChatProfile`
- final_prompt: final prompt after consolidating above information and safety check


In [38]:
@dataclass
class Query:
    """Holds the evolving state of a single user query as it moves through the pipeline."""
    query: str
    user_id: str
    preferred_style: str = "helpful"
    knowledge_context: list[dict] = field(default_factory=list)
    past_relevant_conversation_context: list[str] = field(default_factory=list)
    memory: list[Memory] = field(default_factory=list)
    final_prompt: str = ""


print("Query class defined.")

Query class defined.


#### Chat History Management (mem0, RAG)
This is a raw text chat history memory independent from the memory store seed, for chat memory we define a two system management based on the data given and requriement of mem0:
1. raw text chat history (this section), 
2. chat memory (like `Data/memory_store_seed.json`), a summarized memory that separates into episodic and long term memory, then store in the memory_store_seed

**Description of Chat History Management**  
This layer manages user memory in the raw text format using `mem0` library, it will:
1. store each conversation with a unique `event_id`, and store the `uid`
2. store each question & answer as a trunk
3. allows retrieve using RAG with a threshold, if below threshold, report `not found`



In [39]:
class ChatHistoryManager:
    """Manages raw text chat history using mem0 for storage and retrieval."""

    def __init__(self):
        config = {
            "llm": {
                "provider": "openai",
                "config": {
                    "model": "gpt-5-mini",
                    "api_key": OPENAI_API_KEY,
                },
            },
            "embedder": {
                "provider": "openai",
                "config": {
                    "model": "text-embedding-3-small",
                    "api_key": OPENAI_API_KEY,
                },
            },
            "vector_store": {
                "provider": "qdrant",
                "config": {
                    "collection_name": "chat_history",
                    "embedding_model_dims": 1536,
                },
            },
            "version": "v1.1",
        }
        self.memory = Mem0Memory.from_config(config)

    def add_conversation(self, user_id: str, event_id: str, question: str, answer: str):
        """Store a Q&A pair as a conversation memory."""
        conversation = [
            {"role": "user", "content": question},
            {"role": "assistant", "content": answer},
        ]
        self.memory.add(
            conversation,
            user_id=user_id,
            metadata={"event_id": event_id},
        )

    def search_history(self, query: str, user_id: str, limit: int = 3, threshold: float = 0.5) -> list[str]:
        """Search chat history for relevant past conversations."""
        results = self.memory.search(query=query, user_id=user_id, limit=limit)

        matched = []
        if isinstance(results, dict) and "results" in results:
            entries = results["results"]
        elif isinstance(results, list):
            entries = results
        else:
            return []

        for entry in entries:
            score = entry.get("score", 0)
            if score >= threshold:
                matched.append(entry.get("memory", str(entry)))
        return matched


print("ChatHistoryManager class defined.")

ChatHistoryManager class defined.


#### Knowledge Retrieval (Faiss Database)
**Description**  
This layer use `Data/knowledge_corpus.jsonl` as the source of knowladge, indexed with `category` as filter, will be filtered by user's interest, the content for embedding and semantic search is from `title` and `text`.




In [40]:
class KnowledgeStore:
    """FAISS-backed knowledge retrieval over the financial document corpus."""

    def __init__(self, corpus_path: str):
        self.documents = []
        self.metadata = {}            # int_id → {doc_id, title, category, text}
        self._doc_id_to_int = {}      # doc_id → int_id
        self._int_to_doc_id = {}      # int_id → doc_id

        # Load documents from JSONL
        with open(corpus_path, "r") as f:
            for line in f:
                line = line.strip()
                if line:
                    self.documents.append(json.loads(line))
        print(f"Loaded {len(self.documents)} documents from corpus.")

        # Build FAISS index
        self._build_index()

    def _build_index(self):
        """Embed title+text for each document and build a FAISS IndexIDMap."""
        dim = 1536  # text-embedding-3-small
        base_index = faiss.IndexFlatIP(dim)  # inner product (cosine on normalized vecs)
        self.index = faiss.IndexIDMap(base_index)

        embeddings = []
        ids = []

        for i, doc in enumerate(self.documents):
            int_id = i + 1
            doc_id = doc["doc_id"]

            # Mappings
            self._doc_id_to_int[doc_id] = int_id
            self._int_to_doc_id[int_id] = doc_id
            self.metadata[int_id] = {
                "doc_id": doc_id,
                "title": doc["title"],
                "category": doc["category"],
                "text": doc["text"],
            }

            # Embed title + text
            embed_text = f"{doc['title']}. {doc['text']}"
            vec = get_embedding(embed_text)
            embeddings.append(vec)
            ids.append(int_id)

        # Normalize and add to index
        emb_array = np.array(embeddings, dtype=np.float32)
        faiss.normalize_L2(emb_array)
        self.index.add_with_ids(emb_array, np.array(ids, dtype=np.int64))
        print(f"FAISS index built with {self.index.ntotal} vectors.")

    def search(self, query: str, top_k: int = 3, category_filter: list[str] | None = None) -> list[dict]:
        """Search for relevant documents. Optionally post-filter by category."""
        query_vec = np.array([get_embedding(query)], dtype=np.float32)
        faiss.normalize_L2(query_vec)

        # Search more than top_k if filtering, to have enough after filter
        search_k = self.index.ntotal
        scores, ids = self.index.search(query_vec, search_k)

        results = []
        for score, int_id in zip(scores[0], ids[0]):
            if int_id == -1:
                continue
            meta = self.metadata[int(int_id)]
            if category_filter and meta["category"] not in category_filter:
                continue
            results.append({**meta, "score": float(score)})
            if len(results) >= top_k:
                break
        return results

    def search_personalized(self, query: str, interests: list[str], top_k: int = 3) -> list[dict]:
        """Search with category filtering based on user interests."""
        categories = set()
        for interest in interests:
            key = interest.strip().lower()
            if key in INTEREST_CATEGORY_MAP:
                categories.add(INTEREST_CATEGORY_MAP[key])

        if not categories:
            return self.search(query, top_k=top_k)
        return self.search(query, top_k=top_k, category_filter=list(categories))


print("KnowledgeStore class defined.")

KnowledgeStore class defined.


#### Agents & Tools

##### Profile Agent
**Description**  
The Profile Agent is responsible for reading user-specific profile information and converting it into a safe, structured session context. given uid, extract below information from source

**Source**
- `Data/user_profiles.csv`: `preferred_style`,`segment`,`interests`,`channel`,`consent_personalization`,`profile_summary`
- `Data/interaction_log.csv`: clicked_docs, all `event_id` under same `uid`


In [41]:
def profile_agent(user_id: str, profiles_df: pd.DataFrame, interactions_df: pd.DataFrame) -> ChatProfile:
    """Look up user profile and interaction history to build a ChatProfile.

    Fallback behaviour:
    - Unknown user_id → consent=False, style="helpful", no personalization fields.
    - Known user with consent=N → consent=False, style="helpful", interests cleared.
    """
    row = profiles_df[profiles_df["user_id"] == user_id]

    if row.empty:
        print(f"  User {user_id} not found — returning default profile (no personalization)")
        return ChatProfile(
            user_id=user_id,
            preferred_style="helpful",
            segment="unknown",
            consent_personalization=False,
        )

    row = row.iloc[0]

    # Parse consent flag
    consent = str(row.get("consent_personalization", "N")).strip().upper() == "Y"

    # Parse interests (semicolon-separated)
    raw_interests = [i.strip() for i in str(row.get("interests", "")).split(";") if i.strip()]

    # Get interaction history for this user
    user_interactions = interactions_df[interactions_df["user_id"] == user_id]
    interaction_history = user_interactions.to_dict("records")

    # Collect all clicked_docs across interactions (pipe-separated)
    clicked_docs = []
    for _, inter in user_interactions.iterrows():
        docs = str(inter.get("clicked_docs", "")).split("|")
        clicked_docs.extend([d.strip() for d in docs if d.strip()])

    # If no consent: fall back to "helpful" style, clear personalization fields
    profile = ChatProfile(
        user_id=user_id,
        preferred_style=row.get("preferred_style", "helpful") if consent else "helpful",
        segment=row.get("segment", "unknown"),
        interests=raw_interests if consent else [],
        channel=row.get("channel", ""),
        consent_personalization=consent,
        profile_summary=row.get("profile_summary", "") if consent else "",
        clicked_docs=clicked_docs if consent else [],
        interaction_history=interaction_history,
        memories=[],  # populated later by memory_update_agent
    )

    if not consent:
        print(f"  User {user_id} found but consent=N — using default style, no personalization")
    else:
        print(f"  User {user_id} loaded — style={profile.preferred_style}, interests={profile.interests}")

    return profile


print("profile_agent function defined.")

profile_agent function defined.



#### Memory Update Agent
**Description**  
Load memory all memory class from given `uid` from `Data/memory_store_seed.json` to form the memory for the user, the data

Memory class help to manage the different types of memory, including long term/episodic memory that may have separate expiry dates. Given the different memory type, they should have have different expiry period; and for all effective memory for the same user, the chat session should prioritize them differently, say, prioritize episodic memory over long term memory.

This assignment will only focus on one establish a software architecture for one session leveraging existing data, the memory update agent in this assignment will only focus on the loading, filtering and prioritization of the memory.

Further offline / batch update strategy will be proposed as next step, instead of coding as it requires iteration with real data.

**Full Memory & Chat history Strategy Proposal**
All chat history will be managed in 3 layer, 1 in raw text and 2 in summarized form as memory seed:
1. raw text for current session, store as a vector database, where each trunk is a qustion and answer pair for reference purpose
2. episodic memory, can be saved as per chat session level, utilized summarization tool to focus on interest, preference and topic, store in the memory store seed
3. long term memory, can be generated based on the episodic memory, focus on interest, preference and topic, store in the memory store seed



In [42]:
def memory_update_agent(user_id: str, memory_store: list[Memory], top_k: int = 3) -> list[Memory]:
    """Load, filter expired, and prioritize memories for a given user."""
    # Filter to this user's memories
    user_memories = [m for m in memory_store if m.user_id == user_id]

    # Remove expired memories
    active_memories = [m for m in user_memories if not m.is_expired()]

    expired_count = len(user_memories) - len(active_memories)
    if expired_count > 0:
        print(f"  Filtered out {expired_count} expired memories for {user_id}")

    # Sort by priority_score descending (episodic first, then by confidence)
    active_memories.sort(key=lambda m: m.priority_score(), reverse=True)

    # Return top-k
    selected = active_memories[:top_k]
    print(f"  Loaded {len(selected)} active memories for {user_id}")
    for m in selected:
        print(f"    {m.memory_id} ({m.memory_type}) priority={m.priority_score():.2f}: {m.memory_text[:60]}")
    return selected


print("memory_update_agent function defined.")

memory_update_agent function defined.


#### Retrieval Planner Agent
**Description**  
Given user's query and user's interest from `ChatProfile` to perform RAG and produce knowledge context

The knowledge retrival will take a 3 step approach, this gives personalized context a chance to be select in the final but still based on the relevance at the end:
1. retrieve context given personalized field such as interest and reading history, select top 3
2. retrieve context without filter, select top 3
3. dedup 1 & 2 and select top 3



In [43]:
def retrieval_planner_agent(
    query_obj: Query,
    knowledge_store: KnowledgeStore,
    interests: list[str],
    top_k: int = 3,
) -> list[dict]:
    """Retrieve knowledge context.

    - With consent (interests non-empty): personalized + generic search, dedup, keep ALL unique.
    - Without consent (interests empty): generic search only.
    """
    query_text = query_obj.query

    if interests:
        # Step A: personalized search filtered by user interests
        personalized_results = knowledge_store.search_personalized(query_text, interests, top_k=top_k)
        # Step B: generic search without category filter
        generic_results = knowledge_store.search(query_text, top_k=top_k)
        # Step C: merge, dedup by doc_id, keep ALL unique results (no top-k cap)
        seen_doc_ids = set()
        merged = []
        for result in personalized_results + generic_results:
            if result["doc_id"] not in seen_doc_ids:
                seen_doc_ids.add(result["doc_id"])
                merged.append(result)
        merged.sort(key=lambda x: x["score"], reverse=True)
        final = merged  # keep all deduped results

        print(f"  Retrieval (personalized): {len(personalized_results)} personalized + {len(generic_results)} generic → {len(final)} after dedup")
    else:
        # No-consent fallback: only generic search
        final = knowledge_store.search(query_text, top_k=top_k)
        print(f"  Retrieval (generic only): {len(final)} results")

    for doc in final:
        print(f"    {doc['doc_id']} ({doc['category']}) score={doc['score']:.4f}: {doc['title']}")
    return final


print("retrieval_planner_agent function defined.")

retrieval_planner_agent function defined.


#### Safety Agents
Separate to below two agents

**Description**  
To guardrail the query and context to not include
1. irrelevant questions that is not about financial advise
2. any sensitive information in the context or query

As the provided data seems does not contain sensitive information, guardrail sensitive information including identifier, account, monetary information in this safety agent to exclude them from the prompt.



In [ ]:
def query_safety_agent(query: str) -> dict:
    """Use LLM to check whether the query is financial-domain relevant."""
    messages = [
        {
            "role": "system",
            "content": (
                "You are a safety classifier. Determine if the user query is related to "
                "financial topics (banking, investments, savings, budgeting, treasury, "
                "risk management, insurance, loans, funds, markets). "
                "Make sure the query does not contain any personally identifiable information (PII) or sensitive data."
                "Make sure the query does not contain any harmful or inappropriate content."
                "Respond with ONLY a JSON object: {\"is_safe\": true/false, \"reason\": \"...\"}"
            ),
        },
        {"role": "user", "content": query},
    ]
    response = call_llm(messages)

    try:
        result = json.loads(response)
    except json.JSONDecodeError:
        # If LLM doesn't return valid JSON, try to extract from text
        if "true" in response.lower():
            result = {"is_safe": True, "reason": "Query appears financial-domain relevant."}
        else:
            result = {"is_safe": False, "reason": response}
    return result


# PII patterns for context safety scanning
PII_PATTERNS = [
    (r"\b[STFGstfg]\d{7}[A-Za-z]\b", "[NRIC_REDACTED]"),              # NRIC
    (r"\b\d{10,16}\b", "[ACCOUNT_REDACTED]"),                    # Account numbers
    (r"\b\d{3}[-.\s]?\d{3}[-.\s]?\d{4}\b", "[PHONE_REDACTED]"), # Phone numbers
    (r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b", "[EMAIL_REDACTED]"),  # Email
    (r"\$\d+[\d,.]*\s*(in|from|to)\s+account", "[MONETARY_REF_REDACTED]"),          # Monetary with account ref
]


def context_safety_agent(context_text: str) -> str:
    """Scan context for PII patterns and redact them."""
    cleaned = context_text
    for pattern, replacement in PII_PATTERNS:
        cleaned = re.sub(pattern, replacement, cleaned, flags=re.IGNORECASE)
    return cleaned


# Test safety agents
print("Safety agents defined.")
test_safe = query_safety_agent("Compare money market funds and short-duration bond funds")
print(f"Financial query safe: {test_safe}")
test_unsafe = query_safety_agent("What is the best recipe for chocolate cake?")
print(f"Non-financial query safe: {test_unsafe}")

Safety agents defined.
Financial query safe: {'is_safe': True, 'reason': 'The user asks to compare two investment products—money market funds and short-duration bond funds—so the query clearly concerns financial topics (investments, risk, liquidity).'}
Non-financial query safe: {'is_safe': False, 'reason': 'The query asks for a chocolate cake recipe, which is a cooking/food question and not related to financial topics (banking, investments, insurance, loans, markets, etc.).'}


#### Prompt Construction
**Description**  
Processing all the information in `Query` class pick relevant information from preference, knowledge_context, chat_history_context, safety check then form the final prompt, use `DSPy` library to optimize


In [45]:
# Configure DSPy with OpenAI
lm = dspy.LM("openai/gpt-5-mini", api_key=OPENAI_API_KEY)
dspy.configure(lm=lm)


# ---------------------------------------------------------------------------
# DSPy Signature & Module: Context Relevance Filter
# ---------------------------------------------------------------------------
class ContextRelevanceJudge(dspy.Signature):
    """Given a user query and a numbered list of contexts, identify which contexts
    are relevant to answering the query. Return ONLY the indices of relevant items."""
    query: str = dspy.InputField(desc="The user's financial question")
    contexts: str = dspy.InputField(desc="Numbered contexts, each prefixed with [K<i>] for knowledge or [H<i>] for chat history")
    relevant_indices: str = dspy.OutputField(desc="Comma-separated relevant labels (e.g. 'K0, K2, H0') or 'none' if nothing is relevant")


class ContextRelevanceFilter(dspy.Module):
    """Batch-evaluate all retrieved contexts against the query and keep only relevant ones."""

    def __init__(self):
        super().__init__()
        self.judge = dspy.ChainOfThought(ContextRelevanceJudge)

    def forward(
        self,
        query: str,
        knowledge_contexts: list[dict],
        history_contexts: list[str],
    ) -> tuple[list[dict], list[str]]:
        # Build numbered list for the LLM
        numbered = []
        for i, doc in enumerate(knowledge_contexts):
            numbered.append(f"[K{i}] {doc['title']}: {doc['text'][:300]}")
        for i, ctx in enumerate(history_contexts):
            numbered.append(f"[H{i}] {ctx[:300]}")

        if not numbered:
            return knowledge_contexts, history_contexts

        contexts_text = "\n".join(numbered)
        result = self.judge(query=query, contexts=contexts_text)

        # Parse response
        indices_str = result.relevant_indices.strip().lower()
        if indices_str == "none" or not indices_str:
            print("  Context filter: all contexts judged irrelevant — removing all")
            return [], []

        # Parse labels like K0, K2, H0
        filtered_knowledge = []
        filtered_history = []
        for token in indices_str.replace(",", " ").split():
            token = token.strip().strip("[]")
            if token.startswith("k") and token[1:].isdigit():
                idx = int(token[1:])
                if 0 <= idx < len(knowledge_contexts):
                    filtered_knowledge.append(knowledge_contexts[idx])
            elif token.startswith("h") and token[1:].isdigit():
                idx = int(token[1:])
                if 0 <= idx < len(history_contexts):
                    filtered_history.append(history_contexts[idx])

        print(f"  Context filter: kept {len(filtered_knowledge)}/{len(knowledge_contexts)} knowledge, "
              f"{len(filtered_history)}/{len(history_contexts)} history")
        return filtered_knowledge, filtered_history


context_filter = ContextRelevanceFilter()


# ---------------------------------------------------------------------------
# DSPy Signature & Module: Personalized Financial QA
# ---------------------------------------------------------------------------
class PersonalizedFinancialQA(dspy.Signature):
    """Answer a financial question using the provided context, adapting to the user's communication style."""
    persona_style: str = dspy.InputField(desc="Communication style: concise_technical, empathetic_simple, or executive_brief")
    user_context: str = dspy.InputField(desc="User profile summary and preferences")
    knowledge_context: str = dspy.InputField(desc="Retrieved financial knowledge documents")
    query: str = dspy.InputField(desc="The user's financial question")
    memory_context: str = dspy.InputField(desc="Relevant user memories and past conversation context")
    answer: str = dspy.OutputField(desc="Personalized financial answer grounded in the knowledge context")


class PromptConstructor(dspy.Module):
    """Builds and executes a persona-aware financial QA prompt using DSPy ChainOfThought."""

    def __init__(self):
        super().__init__()
        self.cot = dspy.ChainOfThought(PersonalizedFinancialQA)

    def forward(self, query_obj: Query, profile: ChatProfile) -> str:
        # Build knowledge context string
        knowledge_text = ""
        for i, doc in enumerate(query_obj.knowledge_context, 1):
            knowledge_text += f"[{doc['doc_id']}] {doc['title']}: {doc['text']}\n"
        if not knowledge_text:
            knowledge_text = "No relevant knowledge documents found."

        # Build memory context string
        memory_parts = []
        for m in query_obj.memory:
            memory_parts.append(f"[{m.memory_type}] {m.memory_text}")
        for ctx in query_obj.past_relevant_conversation_context:
            memory_parts.append(f"[chat_history] {ctx}")
        memory_text = "\n".join(memory_parts) if memory_parts else "No prior memory context."

        # Build user context
        user_context = profile.profile_summary if profile.has_consent() else "General user, no personalization."

        result = self.cot(
            persona_style=profile.preferred_style if profile.has_consent() else "helpful",
            user_context=user_context,
            knowledge_context=knowledge_text,
            query=query_obj.query,
            memory_context=memory_text,
        )
        return result.answer


# Standalone prompt builder for the LangGraph flow
PERSONA_INSTRUCTIONS = {
    "concise_technical": "Respond in a concise, technical style. Use bullet points, comparisons, and precise financial terminology. Avoid marketing language.",
    "empathetic_simple": "Respond in a warm, empathetic tone. Use simple language, step-by-step guidance, and encouraging words. Avoid jargon.",
    "executive_brief": "Respond with a short executive summary. Lead with risk flags, then key metrics, then recommendations. Be direct and brief.",
    "helpful": "Respond as a helpful financial assistant. Be clear, accurate, and professional.",
}


def build_final_prompt(query_obj: Query, profile: ChatProfile, max_features: int = 6) -> str:
    """Construct the full system+user prompt from Query and ChatProfile."""
    style = profile.preferred_style if profile.has_consent() else "helpful"
    persona_instruction = PERSONA_INSTRUCTIONS.get(style, PERSONA_INSTRUCTIONS["helpful"])

    # System prompt
    system_parts = [f"You are a personalized financial assistant.\n\n**Style**: {persona_instruction}"]

    if profile.has_consent() and profile.profile_summary:
        system_parts.append(f"\n**User Profile**: {profile.profile_summary}")

    # Knowledge context (already filtered by DSPy ContextRelevanceFilter)
    if query_obj.knowledge_context:
        knowledge_lines = []
        for doc in query_obj.knowledge_context[:max_features]:
            knowledge_lines.append(f"- [{doc['doc_id']}] {doc['title']}: {doc['text']}")
        system_parts.append("\n**Knowledge Context**:\n" + "\n".join(knowledge_lines))

    # Memory context
    memory_lines = []
    for m in query_obj.memory[:max_features]:
        memory_lines.append(f"- [{m.memory_type}] {m.memory_text}")
    # Chat history (already filtered by DSPy ContextRelevanceFilter)
    for ctx in query_obj.past_relevant_conversation_context[:max_features]:
        memory_lines.append(f"- [past_conversation] {ctx}")
    if memory_lines:
        system_parts.append("\n**Memory Context**:\n" + "\n".join(memory_lines))

    system_prompt = "\n".join(system_parts)
    return system_prompt


prompt_constructor = PromptConstructor()
print("DSPy ContextRelevanceFilter, PromptConstructor, and build_final_prompt ready.")

DSPy ContextRelevanceFilter, PromptConstructor, and build_final_prompt ready.



#### Agentic Flow - Orchestration
**Description**
Use LangGraph to form the query orchestration layer. Profile + memory loading happens **once** at session start (`start_session`), outside the graph.

**Session initialization (outside LangGraph):**
1. `start_session(user_id)` → calls Profile Agent + Memory Update Agent → returns `ChatProfile`

**Per-query LangGraph flow:**
1. `init_query` — create `Query` from pre-loaded `ChatProfile`
2. `history_search` — search mem0 chat history for relevant past conversations
3. `retrieval_plan` — FAISS retrieval (personalized + generic if consent, generic-only otherwise), keep all deduped results
4. `safety_check` — LLM query relevance classifier
5. `context_filter` — **DSPy ContextRelevanceFilter** to exclude irrelevant knowledge & history contexts + PII redaction
6. `prompt_build` — construct system prompt with persona instructions
7. `generate` — OpenAI LLM generation
8. `memory_store` — save Q&A to mem0

In [46]:
# =====================================================================
# Session initializer (runs ONCE per chat session, OUTSIDE LangGraph)
# =====================================================================

def start_session(
    user_id: str,
    profiles_df: pd.DataFrame,
    interactions_df: pd.DataFrame,
    memory_store: list[Memory],
    top_k_memories: int = 3,
) -> ChatProfile:
    """Load ChatProfile + memories at session start. Called before any queries."""
    print(f"=== Starting session for {user_id} ===")

    # Step 1: Load profile
    profile = profile_agent(user_id, profiles_df, interactions_df)

    # Step 2: Load & prioritize memories (only if consent)
    if profile.has_consent():
        memories = memory_update_agent(user_id, memory_store, top_k=top_k_memories)
        profile.memories = memories
    else:
        print(f"  No consent — skipping memory loading for {user_id}")
        profile.memories = []

    print(f"=== Session ready for {user_id} (style={profile.preferred_style}, "
          f"consent={profile.has_consent()}, memories={len(profile.memories)}) ===\n")
    return profile


# =====================================================================
# LangGraph State  (profile is now pre-loaded, passed in as input)
# =====================================================================

class RAGState(TypedDict):
    user_id: str
    query_text: str
    chat_profile: Optional[ChatProfile]      # pre-loaded by start_session
    query_obj: Optional[Query]
    knowledge_context: list
    safety_result: Optional[dict]
    final_prompt: str
    answer: str
    chat_history_context: list


# =====================================================================
# LangGraph Node functions
# =====================================================================

def init_query_node(state: RAGState) -> dict:
    """Create the Query object from the pre-loaded ChatProfile."""
    print(f"[1] Initializing query")
    profile = state["chat_profile"]
    query_obj = Query(
        query=state["query_text"],
        user_id=state["user_id"],
        preferred_style=profile.preferred_style,
        memory=profile.memories,
    )
    return {"query_obj": query_obj}


def history_search_node(state: RAGState) -> dict:
    """Search chat history for relevant past conversations."""
    print(f"[2] Chat history search for {state['user_id']}")
    try:
        history = chat_history_manager.search_history(
            state["query_text"], state["user_id"], limit=3
        )
    except Exception as e:
        print(f"  Chat history search failed: {e}")
        history = []
    query_obj = state["query_obj"]
    query_obj.past_relevant_conversation_context = history
    print(f"  Found {len(history)} relevant history entries")
    return {"chat_history_context": history, "query_obj": query_obj}


def retrieval_plan_node(state: RAGState) -> dict:
    """Retrieve relevant knowledge documents."""
    print(f"[3] Knowledge retrieval for query: {state['query_text'][:50]}...")
    profile = state["chat_profile"]
    query_obj = state["query_obj"]
    interests = profile.interests if profile.has_consent() else []
    knowledge = retrieval_planner_agent(query_obj, knowledge_store, interests)
    query_obj.knowledge_context = knowledge
    return {"knowledge_context": knowledge, "query_obj": query_obj}


def safety_check_node(state: RAGState) -> dict:
    """Run query safety check (domain relevance)."""
    print(f"[4] Safety check")
    safety_result = query_safety_agent(state["query_text"])
    print(f"  Query safety: {safety_result}")
    return {"safety_result": safety_result}


def context_filter_node(state: RAGState) -> dict:
    """Use DSPy ContextRelevanceFilter to remove irrelevant contexts, then PII-redact survivors."""
    print(f"[5] Context relevance filter (DSPy)")
    query_obj = state["query_obj"]

    # DSPy batch filtering
    filtered_knowledge, filtered_history = context_filter(
        query=query_obj.query,
        knowledge_contexts=query_obj.knowledge_context,
        history_contexts=query_obj.past_relevant_conversation_context,
    )

    # PII redaction on surviving contexts
    cleaned_knowledge = []
    for doc in filtered_knowledge:
        cleaned_knowledge.append({**doc, "text": context_safety_agent(doc["text"])})
    for m in query_obj.memory:
        m.memory_text = context_safety_agent(m.memory_text)

    query_obj.knowledge_context = cleaned_knowledge
    query_obj.past_relevant_conversation_context = filtered_history
    return {"knowledge_context": cleaned_knowledge, "query_obj": query_obj}


def prompt_build_node(state: RAGState) -> dict:
    """Build the final prompt for LLM generation."""
    print(f"[6] Prompt construction")
    profile = state["chat_profile"]
    query_obj = state["query_obj"]
    max_features = config.get("prompting", {}).get("max_profile_features_in_prompt", 6)
    final_prompt = build_final_prompt(query_obj, profile, max_features=max_features)
    query_obj.final_prompt = final_prompt
    return {"final_prompt": final_prompt, "query_obj": query_obj}


def generate_node(state: RAGState) -> dict:
    """Call LLM to generate the personalized answer."""
    print(f"[7] LLM generation")
    messages = [
        {"role": "system", "content": state["final_prompt"]},
        {"role": "user", "content": state["query_text"]},
    ]
    answer = call_llm(messages)
    return {"answer": answer}


def memory_store_node(state: RAGState) -> dict:
    """Store the Q&A pair in chat history via mem0."""
    print(f"[8] Storing conversation in memory")
    try:
        event_id = f"E{datetime.now().strftime('%Y%m%d%H%M%S')}"
        chat_history_manager.add_conversation(
            state["user_id"], event_id, state["query_text"], state["answer"]
        )
        print(f"  Stored as {event_id}")
    except Exception as e:
        print(f"  Memory store failed: {e}")
    return {}


# =====================================================================
# Safety router
# =====================================================================

def safety_router(state: RAGState) -> str:
    safety = state.get("safety_result", {})
    return "safe" if safety.get("is_safe", False) else "unsafe"


# =====================================================================
# Build the StateGraph
# =====================================================================

def build_rag_graph() -> StateGraph:
    graph = StateGraph(RAGState)

    # Nodes (profile loading is external — no profile_lookup / memory_load here)
    graph.add_node("init_query", init_query_node)
    graph.add_node("history_search", history_search_node)
    graph.add_node("retrieval_plan", retrieval_plan_node)
    graph.add_node("safety_check", safety_check_node)
    graph.add_node("context_filter", context_filter_node)
    graph.add_node("prompt_build", prompt_build_node)
    graph.add_node("generate", generate_node)
    graph.add_node("memory_store", memory_store_node)

    # Edges
    graph.add_edge(START, "init_query")
    graph.add_edge("init_query", "history_search")
    graph.add_edge("history_search", "retrieval_plan")
    graph.add_edge("retrieval_plan", "safety_check")

    # Conditional: safe → context_filter, unsafe → END
    graph.add_conditional_edges(
        "safety_check",
        safety_router,
        {"safe": "context_filter", "unsafe": END},
    )

    graph.add_edge("context_filter", "prompt_build")
    graph.add_edge("prompt_build", "generate")
    graph.add_edge("generate", "memory_store")
    graph.add_edge("memory_store", END)

    return graph


# =====================================================================
# Query runner (takes a pre-loaded ChatProfile)
# =====================================================================

def run_query(chat_profile: ChatProfile, query_text: str) -> str:
    """Run a user query through the RAG pipeline with an already-loaded profile."""
    print(f"\n{'='*60}")
    print(f"Query from {chat_profile.user_id}: {query_text}")
    print(f"{'='*60}")

    initial_state: RAGState = {
        "user_id": chat_profile.user_id,
        "query_text": query_text,
        "chat_profile": chat_profile,
        "query_obj": None,
        "knowledge_context": [],
        "safety_result": None,
        "final_prompt": "",
        "answer": "",
        "chat_history_context": [],
    }

    final_state = rag_app.invoke(initial_state)

    if not final_state.get("answer"):
        safety = final_state.get("safety_result", {})
        return f"I'm sorry, I can only help with financial topics. {safety.get('reason', '')}"

    return final_state["answer"]


# =====================================================================
# Config-driven query runner — reads config_template.json agent flags
# =====================================================================

def run_query_config(
    user_id: str,
    query_text: str,
    config_path: str | None = None,
) -> str:
    """End-to-end query runner controlled by config_template.json.

    Reads the config file and honours the ``agents`` flags:
      - profile_agent:            load user profile (else default profile)
      - memory_update_agent:      load & prioritize memories (else skip)
      - retrieval_planner_agent:  FAISS knowledge retrieval (else skip)
      - safety_agent:             LLM query-relevance gate (else skip)

    Nodes that are always active: init_query, history_search,
    context_filter, prompt_build, generate, memory_store.
    """
    # --- 1. Load config ---
    if config_path is None:
        config_path = os.path.join(DATA_DIR, "config_template.json")
    with open(config_path, "r") as f:
        run_config = json.load(f)

    agents = run_config.get("agents", {})
    use_profile   = agents.get("profile_agent", True)
    use_memory    = agents.get("memory_update_agent", True)
    use_retrieval = agents.get("retrieval_planner_agent", True)
    use_safety    = agents.get("safety_agent", True)

    print(f"\n{'='*60}")
    print(f"[run_query_config] user={user_id}")
    print(f"  agents: profile={use_profile}, memory={use_memory}, "
          f"retrieval={use_retrieval}, safety={use_safety}")
    print(f"  query: {query_text}")
    print(f"{'='*60}")

    # --- 2. Session init (conditional on agent flags) ---
    if use_profile:
        profile = profile_agent(user_id, profiles_df, interactions_df)
    else:
        print("  [profile_agent OFF] using default profile")
        profile = ChatProfile(user_id=user_id)

    if use_memory and profile.has_consent():
        top_k = run_config.get("profile_retrieval", {}).get("top_k_profile_memories", 3)
        profile.memories = memory_update_agent(user_id, all_memories, top_k=top_k)
    else:
        if not use_memory:
            print("  [memory_update_agent OFF] skipping memory loading")
        profile.memories = []

    # --- 3. Build dynamic graph based on config ---
    graph = StateGraph(RAGState)

    # Always-present nodes
    graph.add_node("init_query", init_query_node)
    graph.add_node("history_search", history_search_node)
    graph.add_node("context_filter", context_filter_node)
    graph.add_node("prompt_build", prompt_build_node)
    graph.add_node("generate", generate_node)
    graph.add_node("memory_store", memory_store_node)

    # Conditional nodes
    if use_retrieval:
        graph.add_node("retrieval_plan", retrieval_plan_node)
    if use_safety:
        graph.add_node("safety_check", safety_check_node)

    # --- 4. Wire edges dynamically ---
    graph.add_edge(START, "init_query")
    graph.add_edge("init_query", "history_search")

    prev_node = "history_search"

    if use_retrieval:
        graph.add_edge(prev_node, "retrieval_plan")
        prev_node = "retrieval_plan"

    if use_safety:
        graph.add_edge(prev_node, "safety_check")
        graph.add_conditional_edges(
            "safety_check",
            safety_router,
            {"safe": "context_filter", "unsafe": END},
        )
    else:
        graph.add_edge(prev_node, "context_filter")

    graph.add_edge("context_filter", "prompt_build")
    graph.add_edge("prompt_build", "generate")
    graph.add_edge("generate", "memory_store")
    graph.add_edge("memory_store", END)

    app = graph.compile()

    # --- 5. Run the query ---
    initial_state: RAGState = {
        "user_id": user_id,
        "query_text": query_text,
        "chat_profile": profile,
        "query_obj": None,
        "knowledge_context": [],
        "safety_result": None,
        "final_prompt": "",
        "answer": "",
        "chat_history_context": [],
    }

    final_state = app.invoke(initial_state)

    if not final_state.get("answer"):
        safety = final_state.get("safety_result", {})
        return f"I'm sorry, I can only help with financial topics. {safety.get('reason', '')}"

    return final_state["answer"]


print("LangGraph RAG pipeline defined (profile loading external).")

LangGraph RAG pipeline defined (profile loading external).


### Load Session Data

In [47]:
# Load configuration
with open(os.path.join(DATA_DIR, "config_template.json"), "r") as f:
    config = json.load(f)
print("Config loaded:", json.dumps(config, indent=2))

# Load user profiles
profiles_df = pd.read_csv(os.path.join(DATA_DIR, "user_profiles.csv"))
print(f"\nUser profiles: {len(profiles_df)} users")
print(profiles_df[["user_id", "preferred_style", "segment"]].to_string(index=False))

# Load interaction log
interactions_df = pd.read_csv(os.path.join(DATA_DIR, "interaction_log.csv"))
print(f"\nInteraction log: {len(interactions_df)} events")

# Load memory store seed
with open(os.path.join(DATA_DIR, "memory_store_seed.json"), "r") as f:
    memory_data = json.load(f)
all_memories = [Memory.from_dict(m) for m in memory_data]
print(f"\nMemory store: {len(all_memories)} memories loaded")
for m in all_memories:
    print(f"  {m.memory_id} ({m.user_id}, {m.memory_type}) expired={m.is_expired()}")

# Initialize Knowledge Store (builds FAISS index)
print("\nBuilding knowledge index...")
knowledge_store = KnowledgeStore(os.path.join(DATA_DIR, "knowledge_corpus.jsonl"))

# Initialize Chat History Manager (mem0)
print("\nInitializing chat history manager...")
chat_history_manager = ChatHistoryManager()

# Compile LangGraph (no profile nodes — profile loaded externally via start_session)
print("\nCompiling LangGraph pipeline...")
rag_graph = build_rag_graph()
rag_app = rag_graph.compile()
print("Pipeline ready.")

# --- Start sessions for test users (profile + memory loaded here, before queries) ---
top_k_mem = config.get("profile_retrieval", {}).get("top_k_profile_memories", 3)

print("\n" + "=" * 60)
print("Starting user sessions...")
print("=" * 60)
session_u001 = start_session("U001", profiles_df, interactions_df, all_memories, top_k_mem)
session_u002 = start_session("U002", profiles_df, interactions_df, all_memories, top_k_mem)

Config loaded: {
  "profile_retrieval": {
    "top_k_profile_memories": 3,
    "attribute_filters": [
      "consent_personalization=Y"
    ]
  },
  "prompting": {
    "persona_modes": [
      "concise_technical",
      "empathetic_simple",
      "executive_brief"
    ],
    "max_profile_features_in_prompt": 6
  },
  "agents": {
    "profile_agent": true,
    "retrieval_planner_agent": true,
    "memory_update_agent": true,
    "safety_agent": true
  }
}

User profiles: 3 users
user_id   preferred_style       segment
   U001 concise_technical mass_affluent
   U002 empathetic_simple        retail
   U003   executive_brief corp_treasury

Interaction log: 3 events

Memory store: 3 memories loaded
  M001 (U001, long_term_preference) expired=True
  M002 (U002, long_term_preference) expired=False
  M003 (U003, episodic) expired=True

Building knowledge index...
Loaded 5 documents from corpus.
FAISS index built with 5 vectors.

Initializing chat history manager...

Compiling LangGraph pipelin

### Test Run - Generation

- loading the config_template.json

In [ ]:
# === Test 1: U001 (concise_technical, consent=Y) ===
query_1 = "Compare money market funds and short-duration bond funds"
answer_1 = run_query(session_u001, query_1)

# === Test 2: U002 (empathetic_simple, consent=Y) ===
query_2 = "How do I start an emergency fund"
answer_2 = run_query(session_u002, query_2)

# === Display results side-by-side ===
print("\n" + "=" * 80)
print("COMPARISON: Same system, different personas")
print("=" * 80)

print(f"\n{'─' * 40}")
print(f"U001 (concise_technical) — mass_affluent")
print(f"Query: {query_1}")
print(f"{'─' * 40}")
print(answer_1)

print(f"\n{'─' * 40}")
print(f"U002 (empathetic_simple) — retail")
print(f"Query: {query_2}")
print(f"{'─' * 40}")
print(answer_2)

# === Test 3: Safety rejection (off-topic) ===
print(f"\n{'─' * 40}")
print("TEST: Off-topic query (should be rejected)")
print(f"{'─' * 40}")
answer_3 = run_query(session_u001, "What is the best recipe for chocolate cake?")
print(answer_3)

# === Test 4: Unknown user (no consent, generic RAG only) ===
print(f"\n{'─' * 40}")
print("TEST: Unknown user (no profile → no consent → generic RAG)")
print(f"{'─' * 40}")
session_unknown = start_session("U999", profiles_df, interactions_df, all_memories, top_k_mem)
answer_4 = run_query(session_unknown, "What are money market funds?")
print(answer_4)


### Running Results
============================================================
**Query from U001: Compare money market funds and short-duration bond funds**
============================================================
[1] Initializing query
[2] Chat history search for U001
  Found 0 relevant history entries
[3] Knowledge retrieval for query: Compare money market funds and short-duration bond...
  Retrieval (personalized): 2 personalized + 3 generic → 3 after dedup
    DOC001 (wealth) score=0.7393: Money Market Fund Overview
    DOC002 (wealth) score=0.5581: Bond Fund Basics
    DOC006 (treasury) score=0.3605: Liquidity Coverage Policy Update
[4] Safety check
  Query safety: {'is_safe': True, 'reason': 'The query asks to compare money market funds and short-duration bond funds, which are investment/financial topics (funds, markets, risk and asset allocation).'}
[5] Context relevance filter (DSPy)
  Context filter: kept 2/3 knowledge, 0/0 history
[6] Prompt construction
[7] LLM generation
[8] Storing conversation in memory
  Stored as E20260314223525

============================================================
**Query from U002: How do I start an emergency fund**
============================================================
[1] Initializing query
[2] Chat history search for U002
  Found 0 relevant history entries
[3] Knowledge retrieval for query: How do I start an emergency fund...
  Retrieval (personalized): 2 personalized + 3 generic → 3 after dedup
    DOC004 (retail_banking) score=0.6669: Emergency Fund Planning Guide
    DOC005 (retail_banking) score=0.4513: Budgeting Checklist
    DOC001 (wealth) score=0.2988: Money Market Fund Overview
[4] Safety check
  Query safety: {'is_safe': True, 'reason': 'The query asks how to start an emergency fund, which is a personal finance topic (savings/budgeting).'}
[5] Context relevance filter (DSPy)
  Context filter: kept 3/3 knowledge, 0/0 history
[6] Prompt construction
[7] LLM generation
[8] Storing conversation in memory
  Stored as E20260314223557

================================================================================
**COMPARISON: Same system, different personas**
================================================================================

────────────────────────────────────────
U001 (concise_technical) — mass_affluent
Query: Compare money market funds and short-duration bond funds
────────────────────────────────────────
Summary (one line)
- Money market funds = cash-management vehicles focused on preserving principal and providing liquidity; short-duration bond funds = diversified short-term fixed-income strategies that target higher yield at the cost of modest price/interest-rate risk.

Key structural differences
- Holdings
  - Money market funds: treasury bills, repo, commercial paper, CDs, short-term repos; constrained by SEC Rule 2a‑7 (WAM ≤60 days, WAL ≤120 days for registered MMFs).
  - Short‑duration bond funds: short-term corporates, asset‑backed securities, short Treasuries, municipal notes, floating-rate notes; broader credit and sector mix.
- Duration / interest‑rate sensitivity
  - Money market funds: effective duration effectively near 0 (negligible price sensitivity).
  - Short‑duration bond funds: typical effective duration ~0.5–4 years (price change ≈ −duration × Δyield; e.g., duration 2 → ~−2% if rates rise 1%).
- Credit exposure
  - MMFs: low credit risk (government MMFs minimal; prime MMFs have commercial paper/CP risk but are highly regulated).
  - Short‑duration funds: higher credit and spread risk (can include lower‑rated corporates, MBS, ABS).
- Liquidity and NAV behavior
  - MMFs: daily liquidity; stable NAV mechanics (many retail/government funds target $1 NAV; institutional reforms introduced some floating NAVs).
  - Short‑duration funds: daily liquidity but NAV fluctuates with market prices; potential for short-term capital loss.
- Regulation
  - MMFs: governed by SEC Rule 2a‑7 with maturity, diversification and liquidity rules.
  - Short‑duration funds: regulated as mutual funds/ETFs under standard fund rules (no 2a‑7 constraints).

Return and risk tradeoffs
- Yield drivers
  - MMFs: yield ≈ short‑term policy/overnight rates (virtually no term premium).
  - Short‑duration funds: yield = short rates + term premium + credit spread; typically higher yield than MMFs.
- Expected volatility
  - MMFs: very low volatility; principal preservation is the main objective.
  - Short‑duration funds: low-to-moderate volatility. Can produce negative returns if rates jump or spreads widen.
- Principal risk
  - MMFs: not FDIC insured; small principal risk in extreme credit/liquidity stress.
  - Short‑duration funds: real principal risk from mark‑to‑market losses.

Costs and tax considerations
- Fees: MMFs typically have very low expense ratios; short‑duration bond funds often charge higher fees (0.2%–0.8%+ depending on active management).
- Taxes: both have taxable and municipal (tax‑exempt) variants. Short‑duration muni funds can offer tax-exempt income with similar tradeoffs.

Typical use cases
- Money market fund: cash parking, operating cash, emergency fund, settlement account (priority = liquidity + capital preservation).
- Short‑duration bond fund: short- to intermediate-term cash needs (3–24 months up to a few years), seeking incremental yield and willing to accept modest NAV volatility.

Decision checklist
- If priority = preserve principal + instant liquidity → use money market fund.
- If priority = higher current income and you can accept modest price volatility/short holding horizon → consider short‑duration bond fund.
- Consider investment horizon, interest‑rate outlook, credit tolerance, tax status, and liquidity needs.

Concise example
- Scenario: rates rise 1%:
  - Money market fund: yield on new cash increases; NAV essentially unchanged.
  - Short‑duration fund with duration 2: NAV ≈ −2% price move (offset over time by higher reinvestment yields).

If you want, I can:
- Compare specific fund types (government vs prime MMF; ultra‑short vs short‑duration bond ETFs),
- Run a scenario with assumed duration/spread values to show expected return/volatility.

────────────────────────────────────────
U002 (empathetic_simple) — retail
Query: How do I start an emergency fund
────────────────────────────────────────
Great question — starting an emergency fund is one of the smartest things you can do. Here’s a simple, gentle step-by-step plan you can follow right away.

1) Figure out your essential monthly expenses
- Add up things you must pay each month: rent/mortgage, utilities, groceries, insurance, minimum debt payments, phone, transportation.  
- Example: if those add to $2,500/month, that’s your baseline.

2) Set a target
- Common goal: 3–6 months of essential expenses. So with $2,500/month, a 3-month fund = $7,500; 6 months = $15,000.  
- If that feels too big, start smaller: aim for a starter cushion of $500–$1,000 while you build toward your main goal.

3) Choose a safe, liquid place to keep it
- Best choices: a high-yield savings account or a bank money market account that’s FDIC-insured. They keep money accessible and safe.  
- Note: brokerage “money market funds” are often liquid but are not FDIC-insured — they’re fine for liquidity but technically carry different risk.

4) Automate saving
- Set up an automatic transfer from your checking to the emergency account every payday. Treat it like a bill.  
- Even small, regular amounts work. Example: if your goal is $7,500 and you can save $300/month, you’ll reach it in 25 months. If you can save $600/month, you’ll reach it in 12.5 months.

5) Build using simple strategies
- Start with a small “starter” target so you have immediate protection.  
- Put windfalls (tax refunds, bonuses) into the fund.  
- Use rounding or “spare change” apps, or increase contributions when expenses drop.

6) Keep it for true emergencies only
- Define what counts (job loss, medical emergency, major car/home repairs). Avoid using it for planned expenses or everyday wants.  
- Keep the account separate and not too easy to spend (a different online bank or no linked debit card helps).

7) Replenish and review
- If you use the fund, make a plan to rebuild it quickly.  
- Review needs each year (family changes, job risk, new expenses) and adjust your target.

Quick tips if you have debt or irregular income
- If you have high-interest debt, split your approach: build a small starter fund ($500–$1,000), while also paying extra toward high-interest debt.  
- If income varies, base your target on average essential expenses and aim to save a percentage of each paycheque.

If you’d like, I can:
- Help you calculate your target if you tell me your monthly essentials, or  
- Suggest a monthly automatic amount based on a timeline you prefer.

You’ve already taken the important first step by asking — I’m happy to help you make a simple plan. Which part would you like help with next?

────────────────────────────────────────
**TEST: Off-topic query (should be rejected)**
────────────────────────────────────────

============================================================
**Query from U001: What is the best recipe for chocolate cake?**
============================================================
[1] Initializing query
[2] Chat history search for U001
  Found 0 relevant history entries
[3] Knowledge retrieval for query: What is the best recipe for chocolate cake?...
  Retrieval (personalized): 2 personalized + 3 generic → 4 after dedup
    DOC004 (retail_banking) score=0.0515: Emergency Fund Planning Guide
    DOC005 (retail_banking) score=0.0242: Budgeting Checklist
    DOC002 (wealth) score=0.0075: Bond Fund Basics
    DOC001 (wealth) score=-0.0665: Money Market Fund Overview
[4] Safety check
  Query safety: {'is_safe': False, 'reason': 'The query asks for a chocolate cake recipe (cooking/baking), which is unrelated to financial topics like banking, investments, insurance, loans, markets, or risk management.'}
I'm sorry, I can only help with financial topics. The query asks for a chocolate cake recipe (cooking/baking), which is unrelated to financial topics like banking, investments, insurance, loans, markets, or risk management.

────────────────────────────────────────
**TEST: Unknown user (no profile → no consent → generic RAG)**
────────────────────────────────────────
=== Starting session for U999 ===
  User U999 not found — returning default profile (no personalization)
  No consent — skipping memory loading for U999
=== Session ready for U999 (style=helpful, consent=False, memories=0) ===


============================================================
**Query from U999: What are money market funds?**
============================================================
[1] Initializing query
[2] Chat history search for U999
  Found 0 relevant history entries
[3] Knowledge retrieval for query: What are money market funds?...
  Retrieval (generic only): 3 results
    DOC001 (wealth) score=0.7786: Money Market Fund Overview
    DOC002 (wealth) score=0.4392: Bond Fund Basics
    DOC006 (treasury) score=0.2983: Liquidity Coverage Policy Update
[4] Safety check
  Query safety: {'is_safe': True, 'reason': 'The query asks about money market funds, which are an investment/financial product (funds, markets, investments).'}
[5] Context relevance filter (DSPy)
  Context filter: kept 1/3 knowledge, 0/0 history
[6] Prompt construction
[7] LLM generation
[8] Storing conversation in memory
  Stored as E20260314223629
A money market fund is a pooled mutual fund that invests in very short-term, high-quality debt instruments to provide liquidity, capital preservation, and modest income. They’re commonly used as a cash-management vehicle rather than for long‑term growth.

Key points
- What they hold: short-duration instruments such as Treasury bills, repurchase agreements, commercial paper, and bank certificates of deposit.
- Objective: preserve principal, provide daily liquidity, and earn a yield slightly above bank checking accounts (varies with short-term interest rates).
- How they work: professional managers maintain a diversified portfolio and try to keep the net asset value (NAV) stable (many funds aim for a $1.00 NAV per share).
- Types: government (mostly U.S. Treasuries and agencies), prime (broader short-term corporate paper), and tax-exempt (municipal securities for tax-free income).
- Key metrics to watch: 7-day yield (commonly reported), expense ratio, credit quality of holdings, and liquidity rules.
- Safety and risks: generally low risk but not risk-free — funds are not FDIC-insured. Risks include credit risk, interest-rate risk, liquidity risk, and the small chance of “breaking the buck” (NAV falling below $1), which happened rarely and led to regulatory reforms after 2008.
- When to use them: emergency funds, short-term cash parking, brokerage sweep accounts, or as a place to keep proceeds awaiting reinvestment.

If you’d like, I can compare specific money market funds, estimate potential current yields, or show how they compare to high-yield savings accounts or short-term Treasury bills.

In [48]:

# === Test 5: run_query_config (config-driven, all agents ON) ===
print(f"\n{'─' * 40}")
print("TEST: run_query_config (config-driven pipeline)")
print(f"{'─' * 40}")
answer_5 = run_query_config("U001", "Compare money market funds and short-duration bond funds")
print(answer_5)


────────────────────────────────────────
TEST: run_query_config (config-driven pipeline)
────────────────────────────────────────

[run_query_config] user=U001
  agents: profile=True, memory=True, retrieval=True, safety=True
  query: Compare money market funds and short-duration bond funds
  User U001 loaded — style=concise_technical, interests=['wealth management', 'ETF']
  Filtered out 1 expired memories for U001
  Loaded 0 active memories for U001
[1] Initializing query
[2] Chat history search for U001
  Found 0 relevant history entries
[3] Knowledge retrieval for query: Compare money market funds and short-duration bond...
  Retrieval (personalized): 2 personalized + 3 generic → 3 after dedup
    DOC001 (wealth) score=0.7395: Money Market Fund Overview
    DOC002 (wealth) score=0.5581: Bond Fund Basics
    DOC006 (treasury) score=0.3605: Liquidity Coverage Policy Update
[4] Safety check
  Query safety: {'is_safe': True, 'reason': 'The query requests a comparison of two types of in

## Code Snippet for Offline Memory Update (Not-to-run in this notebook)

### Examples
* **file**: ../Data/memory_store_seed.json

### Episodic Memory Update
* **Input**: event_id
* **Source**: all conversation in the chat history with same event_id
* **Summary focus**: preference, interest, topic, and communication history
* **Output**: per session episodic memroy with confidence score, update date and expiry period

### Long-term Memory Update
* **Input**: date_from
* **Source**: all episodic memory after `date_from`
* **Summary focus**: preference, interest, topic, and communication history
* **Output**: long term memroy with confidence score, update date and expiry period


In [ ]:
MEMORY_SEED_PATH = "some/path/memory_store_seed.json"
EPISODIC_EXPIRY_DAYS = 30
LONG_TERM_EXPIRY_DAYS = 180

# =====================================================================
# Episodic Memory Update
# =====================================================================

def build_episodic_memory(
    user_id: str,
    event_id: str,
    chat_history_manager: ChatHistoryManager,
) -> Memory:
    # --- Step 1: Retrieve all conversation turns for this event_id ---
    raw_conversations = chat_history_manager.memory.search(
        query="*",                          # broad match
        user_id=user_id,
        limit=50,                           # upper bound per session
    )

    # Filter to only the turns tagged with this event_id
    session_turns = [
        entry for entry in raw_conversations
        if entry.get("metadata", {}).get("event_id") == event_id
    ]

    # Flatten into a single text block for summarisation
    conversation_text = "\n".join(
        turn.get("memory", str(turn)) for turn in session_turns
    )

    # --- Step 2: LLM-based summarisation focused on user signals ---
    summary_prompt = [
        {
            "role": "system",
            "content": (
                "You are a memory-extraction assistant. Given a chat transcript, "
                "produce a concise summary (1-3 sentences) capturing the user's: "
                "preferences, interests, topics discussed, and communication style. "
                "Also output a confidence score (0.0-1.0) reflecting how clearly "
                "these signals appear in the conversation. "
                "Respond as JSON: {\"summary\": \"...\", \"confidence\": 0.XX}"
            ),
        },
        {"role": "user", "content": conversation_text},
    ]
    llm_response = call_llm(summary_prompt)
    result = json.loads(llm_response)       # {"summary": "...", "confidence": 0.XX}

    # --- Step 3: Build Memory object ---
    memory_id = f"ME_{user_id}_{event_id}"  # ME = Memory Episodic
    episodic_memory = Memory(
        memory_id=memory_id,
        user_id=user_id,
        memory_type="episodic",
        memory_text=result["summary"],
        confidence=result["confidence"],
        expiry_days=EPISODIC_EXPIRY_DAYS,
        last_update=datetime.now().strftime("%Y-%m-%d"),
    )
    return episodic_memory


# =====================================================================
# Long-term Memory Update
# =====================================================================

def build_long_term_memory(
    user_id: str,
    date_from: str,
    memory_store: list[Memory],
) -> Memory:
    # --- Step 1: Collect qualifying episodic memories ---
    cutoff = datetime.strptime(date_from, "%Y-%m-%d")

    recent_episodic = [
        m for m in memory_store
        if m.user_id == user_id
        and m.memory_type == "episodic"
        and datetime.strptime(m.last_update, "%Y-%m-%d") >= cutoff
        and not m.is_expired()
    ]

    # Sort by date (newest first) for recency-weighted summarisation
    recent_episodic.sort(
        key=lambda m: datetime.strptime(m.last_update, "%Y-%m-%d"),
        reverse=True,
    )

    # Combine episodic texts into a single block
    episodic_text = "\n".join(
        f"[{m.last_update}] {m.memory_text}" for m in recent_episodic
    )

    # --- Step 2: LLM-based consolidation ---
    consolidation_prompt = [
        {
            "role": "system",
            "content": (
                "You are a memory-consolidation assistant. Given a list of "
                "time-stamped episodic user memories, produce a single long-term "
                "summary (2-4 sentences) that captures enduring patterns in the "
                "user's: preferences, interests, topics of concern, and "
                "communication style. Discard one-off or contradicted signals. "
                "Also output a confidence score (0.0-1.0) reflecting how "
                "consistent and strong these patterns are. "
                "Respond as JSON: {\"summary\": \"...\", \"confidence\": 0.XX}"
            ),
        },
        {"role": "user", "content": episodic_text},
    ]
    llm_response = call_llm(consolidation_prompt)
    result = json.loads(llm_response)       # {"summary": "...", "confidence": 0.XX}

    # --- Step 3: Build Memory object ---
    memory_id = f"MLT_{user_id}_{date_from}"  # MLT = Memory Long Term
    long_term_memory = Memory(
        memory_id=memory_id,
        user_id=user_id,
        memory_type="long_term_preference",
        memory_text=result["summary"],
        confidence=result["confidence"],
        expiry_days=LONG_TERM_EXPIRY_DAYS,
        last_update=datetime.now().strftime("%Y-%m-%d"),
    )
    return long_term_memory


# =====================================================================
# Batch runner — orchestrates both updates and persists to JSON
# =====================================================================

def run_offline_memory_update(
    user_id: str,
    event_ids: list[str],
    date_from: str,
    chat_history_manager: ChatHistoryManager,
    memory_store: list[Memory],
    output_path: str = MEMORY_SEED_PATH,
):
    # --- Phase 1: Episodic memory for each new session ---
    for event_id in event_ids:
        episodic_mem = build_episodic_memory(user_id, event_id, chat_history_manager)
        memory_store.append(episodic_mem)
        print(f"  [episodic] {episodic_mem.memory_id} | conf={episodic_mem.confidence:.2f} | {episodic_mem.memory_text[:80]}")

    # --- Phase 2: Long-term consolidation from recent episodic ---
    long_term_mem = build_long_term_memory(user_id, date_from, memory_store)
    memory_store.append(long_term_mem)
    print(f"  [long_term] {long_term_mem.memory_id} | conf={long_term_mem.confidence:.2f} | {long_term_mem.memory_text[:80]}")

    # --- Phase 3: Persist to JSON ---
    serialised = [
        {
            "memory_id": m.memory_id,
            "user_id": m.user_id,
            "memory_type": m.memory_type,
            "memory_text": m.memory_text,
            "confidence": m.confidence,
            "expiry_days": m.expiry_days,
            "last_update": m.last_update,
        }
        for m in memory_store
    ]
    with open(output_path, "w") as f:
        json.dump(serialised, f, indent=2)
    print(f"  Memory store saved to {output_path} ({len(serialised)} entries)")

